# V-FLoRA GLUE High-Round Monitoring

Lightweight monitoring notebook for the high-round GLUE RoBERTa client-count runs. It scans completed `round_config.json` files only, so it can update plots from partial segmented runs without loading model checkpoints.

The standalone V-FLoRA default is the QNLI RoBERTa e20/r150 client-count workflow. Other GLUE task configurations remain available but disabled until their result folders are present.

In [1]:
from pathlib import Path
import json
import math
import re

from IPython.display import display
import pandas as pd
import plotly.express as px

BASE_DIR = Path('.')
OUTPUT_DIR = BASE_DIR / 'tuning_analysis_outputs' / 'glue_high_round_monitoring'
PLOT_DIR = BASE_DIR / 'plots_epoch_round_tuning' / 'glue_high_round_monitoring'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

ENABLE_RTE = False
ENABLE_MRPC = False
ENABLE_COLA = False
ENABLE_QNLI = True

RUN_DIR_RE = re.compile(
    r'^tuning-(?P<method>flora|nonlinear_flora|linear_flora_cumulative|ffa)-(?P<dataset>.+)-'
    r'(?P<model>tinyllama|llama|llama-7b|roberta-base)-(?P<setting>homo|heter)-'
    r'e(?P<epochs>\d+)-r(?P<rounds>\d+)$'
)

METHOD_ORDER = ['Linear FLoRA', 'Linear FLoRA Cumulative', 'Nonlinear FLoRA Cumulative', 'Nonlinear FFA']
MODEL_LABELS = {
    'tinyllama': 'TinyLlama',
    'llama': 'Llama-7B',
    'llama-7b': 'Llama-7B',
    'roberta-base': 'RoBERTa-base',
}
SETTING_LABELS = {'homo': 'Homo', 'heter': 'Heter'}
DATASET_LABELS = {
    'rte': 'RTE',
    'rte_stratified': 'RTE stratified',
    'rte_dirichlet': 'RTE Dirichlet',
    'mrpc': 'MRPC',
    'mrpc_stratified': 'MRPC stratified',
    'cola': 'CoLA',
    'cola_stratified': 'CoLA stratified',
    'qnli': 'QNLI',
    'qnli_stratified': 'QNLI stratified',
}
DEFAULT_METHOD_LABELS = {
    'flora': 'Linear FLoRA',
    'nonlinear_flora': 'Nonlinear FLoRA Cumulative',
    'linear_flora_cumulative': 'Linear FLoRA Cumulative',
    'ffa': 'Nonlinear FFA',
}

RAW_COLUMNS = [
    'Task', 'Run source', 'Method key', 'Method', 'Dataset', 'Dataset label',
    'Model key', 'Model', 'Setting key', 'Setting', 'Local epochs', 'Config rounds',
    'Seed', 'Client count', 'Client count label', 'Round id', 'Communication round',
    'Metric key', 'Metric label', 'Metric value', 'Accuracy (%)', 'Selected clients',
    'Round stacked rank', 'Cumulative or global rank', 'Rank semantics',
    'Local metrics path', 'Config path',
]
SUMMARY_COLUMNS = [
    'Task', 'Method key', 'Method', 'Dataset', 'Dataset label', 'Model key', 'Model',
    'Setting key', 'Setting', 'Local epochs', 'Config rounds', 'Client count',
    'Client count label', 'Communication round', 'Round id', 'Mean metric', 'Std metric',
    'Mean accuracy (%)', 'Seed count', 'Result source',
]
LATEST_COLUMNS = [
    'Task', 'Method', 'Client count', 'Communication round', 'Mean metric',
    'Std metric', 'Mean accuracy (%)', 'Seed count',
]

TASKS = {
    'rte': {
        'slug': 'rte',
        'title': 'RTE monitored aggregated validation accuracy',
        'task_label': 'RTE',
        'metric_key': 'accuracy',
        'metric_label': 'RTE validation accuracy (%)',
        'metric_scale': 100.0,
        'metric_plot_name': 'aggregated_validation_accuracy',
        'local_epochs': 20,
        'config_rounds': 150,
        'datasets': {'rte', 'rte_stratified', 'rte_dirichlet'},
        'sources': [
            {
                'name': 'monitored client-count run',
                'root': BASE_DIR / 'epoch_round_tuning_rte_client_count_monitored',
                'method_labels': {
                    'flora': 'Linear FLoRA Cumulative',
                    'nonlinear_flora': 'Nonlinear FLoRA Cumulative',
                    'ffa': 'Nonlinear FFA',
                },
            },
            {
                'name': 'clean normal FLoRA run',
                'root': BASE_DIR / 'epoch_round_tuning_rte_normal_flora',
                'method_labels': {'flora': 'Linear FLoRA'},
            },
        ],
    },
    'mrpc': {
        'slug': 'mrpc',
        'title': 'MRPC monitored aggregated validation accuracy',
        'task_label': 'MRPC',
        'metric_key': 'accuracy',
        'metric_label': 'MRPC validation accuracy (%)',
        'metric_scale': 100.0,
        'metric_plot_name': 'aggregated_validation_accuracy',
        'local_epochs': 20,
        'config_rounds': 150,
        'datasets': {'mrpc', 'mrpc_stratified'},
        'sources': [
            {
                'name': 'mrpc client-count run',
                'root': BASE_DIR / 'epoch_round_tuning_mrpc_client_count',
                'method_labels': {'flora': 'Linear FLoRA', 'ffa': 'Nonlinear FFA'},
            },
        ],
    },
    'cola': {
        'slug': 'cola',
        'title': 'CoLA monitored Matthews correlation',
        'task_label': 'CoLA',
        'metric_key': 'matthews_correlation',
        'metric_label': 'Matthews correlation',
        'metric_scale': 1.0,
        'metric_plot_name': 'matthews_correlation',
        'local_epochs': 20,
        'config_rounds': 150,
        'datasets': {'cola', 'cola_stratified'},
        'sources': [
            {
                'name': 'cola client-count run',
                'root': BASE_DIR / 'epoch_round_tuning_cola_client_count',
                'method_labels': {'flora': 'Linear FLoRA', 'ffa': 'Nonlinear FFA'},
            },
        ],
    },
    'qnli': {
        'slug': 'qnli',
        'title': 'QNLI monitored aggregated validation accuracy',
        'task_label': 'QNLI',
        'metric_key': 'accuracy',
        'metric_label': 'QNLI validation accuracy (%)',
        'metric_scale': 100.0,
        'metric_plot_name': 'aggregated_validation_accuracy',
        'local_epochs': 20,
        'config_rounds': 150,
        'datasets': {'qnli', 'qnli_stratified'},
        'sources': [
            {
                'name': 'qnli e20r150 client-count run',
                'root': BASE_DIR / 'epoch_round_tuning_qnli_client_count_e20_r150',
                'method_labels': {'flora': 'Linear FLoRA', 'ffa': 'Nonlinear FFA'},
            },
        ],
    },
}

print(f'Outputs will be written under {OUTPUT_DIR} and {PLOT_DIR}.')

Outputs will be written under tuning_analysis_outputs/glue_high_round_monitoring and plots_epoch_round_tuning/glue_high_round_monitoring.


In [2]:
def _client_sort_key(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return value


def _nice_tick_step(max_value, target_ticks=9):
    if max_value is None or pd.isna(max_value) or max_value <= 0:
        return 1
    raw_step = max_value / max(target_ticks, 1)
    if raw_step <= 1:
        return 1
    magnitude = 10 ** math.floor(math.log10(raw_step))
    for multiplier in (1, 2, 5, 10):
        step = multiplier * magnitude
        if step >= raw_step:
            return step
    return 10 * magnitude


def _format_facet_labels(fig):
    fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))


def _save_plot(fig, output_stem, png_warnings):
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(output_stem.with_suffix('.html'), include_plotlyjs='cdn')
    try:
        fig.write_image(output_stem.with_suffix('.png'), scale=2)
    except Exception as exc:
        message = str(exc).strip() or type(exc).__name__
        png_warnings.append(message.splitlines()[0])


def _parse_run_dir(run_dir, task_config, source):
    match = RUN_DIR_RE.match(run_dir.name)
    if not match:
        return None
    info = match.groupdict()
    if info['dataset'] not in task_config['datasets']:
        return None
    local_epochs = int(info['epochs'])
    config_rounds = int(info['rounds'])
    if local_epochs != task_config['local_epochs'] or config_rounds != task_config['config_rounds']:
        return None
    method_key = info['method']
    method_labels = source.get('method_labels', {})
    return {
        'Task': task_config['task_label'],
        'Run source': source['name'],
        'Method key': method_key,
        'Method': method_labels.get(method_key, DEFAULT_METHOD_LABELS.get(method_key, method_key)),
        'Dataset': info['dataset'],
        'Dataset label': DATASET_LABELS.get(info['dataset'], info['dataset']),
        'Model key': info['model'],
        'Model': MODEL_LABELS.get(info['model'], info['model']),
        'Setting key': info['setting'],
        'Setting': SETTING_LABELS.get(info['setting'], info['setting']),
        'Local epochs': local_epochs,
        'Config rounds': config_rounds,
    }


def load_round_metrics(task_config):
    rows = []
    metric_key = task_config['metric_key']
    metric_scale = float(task_config.get('metric_scale', 1.0))
    for source in task_config['sources']:
        root = source['root']
        for config_path in sorted(root.glob('tuning-*/seed*/[0-9]*/[0-9]*/round_config.json')):
            round_dir = config_path.parent
            client_dir = round_dir.parent
            seed_dir = client_dir.parent
            run_dir = seed_dir.parent
            info = _parse_run_dir(run_dir, task_config, source)
            if info is None:
                continue
            seed_text = seed_dir.name.removeprefix('seed')
            if not seed_text.isdigit():
                continue
            with config_path.open() as handle:
                config = json.load(handle)
            raw_metric = config.get(metric_key)
            if raw_metric is None:
                continue
            round_id = int(config.get('epoch', round_dir.name))
            accuracy = config.get('accuracy')
            row = dict(info)
            row.update(
                {
                    'Seed': int(seed_text),
                    'Client count': int(client_dir.name),
                    'Client count label': str(client_dir.name),
                    'Round id': round_id,
                    'Communication round': round_id + 1,
                    'Metric key': metric_key,
                    'Metric label': task_config['metric_label'],
                    'Metric value': float(raw_metric) * metric_scale,
                    'Accuracy (%)': float(accuracy) * 100 if accuracy is not None else None,
                    'Selected clients': ', '.join(map(str, config.get('selected_clients', []))),
                    'Round stacked rank': config.get('round_stacked_r'),
                    'Cumulative or global rank': config.get('cumulative_or_global_r'),
                    'Rank semantics': config.get('rank_semantics'),
                    'Local metrics path': config.get('local_metrics_path'),
                    'Config path': str(config_path),
                }
            )
            rows.append(row)
    if not rows:
        return pd.DataFrame(columns=RAW_COLUMNS)
    return pd.DataFrame(rows).sort_values(
        ['Task', 'Client count', 'Method', 'Seed', 'Round id']
    ).reset_index(drop=True)


def summarize_round_metrics(raw):
    if raw.empty:
        return pd.DataFrame(columns=SUMMARY_COLUMNS)
    summary = (
        raw.groupby(
            [
                'Task', 'Method key', 'Method', 'Dataset', 'Dataset label', 'Model key', 'Model',
                'Setting key', 'Setting', 'Local epochs', 'Config rounds', 'Client count',
                'Client count label', 'Communication round', 'Round id',
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            **{
                'Mean metric': ('Metric value', 'mean'),
                'Std metric': ('Metric value', 'std'),
                'Mean accuracy (%)': ('Accuracy (%)', 'mean'),
                'Seed count': ('Seed', 'nunique'),
                'Result source': ('Config path', lambda values: 'round_config.json'),
            }
        )
        .sort_values(['Client count', 'Method', 'Communication round'])
        .reset_index(drop=True)
    )
    summary['Std metric'] = summary['Std metric'].fillna(0.0)
    return summary


def latest_round_table(summary):
    if summary.empty:
        return pd.DataFrame(columns=LATEST_COLUMNS)
    idx = summary.groupby(['Task', 'Method', 'Client count'])['Communication round'].idxmax()
    return (
        summary.loc[idx, LATEST_COLUMNS]
        .sort_values(['Client count', 'Method'])
        .reset_index(drop=True)
    )


def make_metric_curve(summary, task_config):
    if summary.empty:
        return None
    data = summary.sort_values(['Client count', 'Method', 'Communication round']).copy()
    hover_data = {
        'Std metric': ':.4f',
        'Seed count': True,
        'Round id': True,
        'Result source': True,
    }
    if data['Mean accuracy (%)'].notna().any() and task_config['metric_key'] != 'accuracy':
        hover_data['Mean accuracy (%)'] = ':.2f'
    method_order = [method for method in METHOD_ORDER if method in set(data['Method'])]
    client_order = [str(value) for value in sorted(data['Client count label'].unique(), key=_client_sort_key)]
    fig = px.line(
        data,
        x='Communication round',
        y='Mean metric',
        color='Method',
        facet_col='Client count label',
        markers=True,
        template='plotly_white',
        title=task_config['title'],
        labels={
            'Mean metric': task_config['metric_label'],
            'Client count label': 'Clients',
        },
        category_orders={'Method': method_order, 'Client count label': client_order},
        hover_data=hover_data,
    )
    _format_facet_labels(fig)
    fig.update_layout(
        width=1250,
        height=540,
        title={
            'x': 0.5,
            'xanchor': 'center',
            'y': 0.985,
            'yanchor': 'top',
            'font': {'size': 18},
        },
        margin={'l': 82, 'r': 36, 't': 118, 'b': 118},
        legend={
            'orientation': 'h',
            'x': 0.5,
            'xanchor': 'center',
            'y': -0.20,
            'yanchor': 'top',
        },
        legend_title_text='',
        hovermode='x unified',
    )
    fig.update_xaxes(
        showgrid=True,
        gridcolor='#d9d9d9',
        gridwidth=1,
        automargin=True,
        title_standoff=18,
        tickangle=-35,
        tickmode='linear',
        dtick=_nice_tick_step(data['Communication round'].max(), target_ticks=8),
        tick0=1,
    )
    fig.update_yaxes(
        showgrid=True,
        gridcolor='#d9d9d9',
        gridwidth=1,
        automargin=True,
        title_standoff=18,
    )
    return fig


def monitor_task(task_key):
    task_config = TASKS[task_key]
    slug = task_config['slug']
    raw = load_round_metrics(task_config)
    summary = summarize_round_metrics(raw)
    latest = latest_round_table(summary)

    raw.to_csv(OUTPUT_DIR / f'{slug}_round_metrics_raw.csv', index=False)
    summary.to_csv(OUTPUT_DIR / f'{slug}_round_metrics_summary.csv', index=False)
    latest.to_csv(OUTPUT_DIR / f'{slug}_latest_rounds.csv', index=False)

    print(
        f"{task_config['task_label']}: loaded {len(raw)} metric rows "
        f"from {raw['Config path'].nunique() if not raw.empty else 0} round_config files."
    )
    if latest.empty:
        print(f"No {task_config['task_label']} {task_config['metric_label']} rows found yet.")
    else:
        display(latest.round(4))

    png_warnings = []
    fig = make_metric_curve(summary, task_config)
    if fig is not None:
        _save_plot(fig, PLOT_DIR / f"{slug}_{task_config['metric_plot_name']}", png_warnings)
        fig.show()
    if png_warnings:
        print(f"PNG export skipped or incomplete; HTML plot was written. First warning: {png_warnings[0]}")
    return raw, summary, latest, fig

## RTE Monitoring (Disabled By Default)

RTE monitoring is available for local use when the corresponding result folders are present, but it is skipped by default for handoff.

In [3]:
if ENABLE_RTE:
    rte_raw, rte_summary, rte_latest, rte_fig = monitor_task('rte')
else:
    print('RTE monitoring disabled. Set ENABLE_RTE = True in the setup cell to enable it.')

RTE monitoring disabled. Set ENABLE_RTE = True in the setup cell to enable it.


## MRPC Monitoring (Disabled By Default)

MRPC monitoring is available for local use when the corresponding result folders are present, but it is skipped by default for handoff.

In [4]:
if ENABLE_MRPC:
    mrpc_raw, mrpc_summary, mrpc_latest, mrpc_fig = monitor_task('mrpc')
else:
    print('MRPC monitoring disabled. Set ENABLE_MRPC = True in the setup cell to enable it.')

MRPC monitoring disabled. Set ENABLE_MRPC = True in the setup cell to enable it.


## CoLA Monitoring

CoLA monitoring is available but disabled by default. Enable it after the corresponding vflora result folder is available.

In [5]:
if ENABLE_COLA:
    cola_raw, cola_summary, cola_latest, cola_fig = monitor_task('cola')
else:
    print('CoLA monitoring disabled. Set ENABLE_COLA = True in the setup cell to enable it.')

CoLA monitoring disabled. Set ENABLE_COLA = True in the setup cell to enable it.


## QNLI Monitoring

QNLI is enabled by default. This section scans `epoch_round_tuning_qnli_client_count_e20_r150` for completed `round_config.json` files and plots validation accuracy by communication round, method, and client count. It can be rerun while segmented jobs are active; each completed round appears as soon as its metadata is written.

In [6]:
if ENABLE_QNLI:
    qnli_raw, qnli_summary, qnli_latest, qnli_fig = monitor_task('qnli')
else:
    print('QNLI monitoring disabled. Set ENABLE_QNLI = True in the setup cell to enable it.')

QNLI: loaded 0 metric rows from 0 round_config files.
No QNLI QNLI validation accuracy (%) rows found yet.


## Output Files

The notebook writes monitoring CSVs and Plotly HTML files under task-specific filenames in the monitoring output folders configured in the setup cell.

In [7]:
print(f'CSV output directory: {OUTPUT_DIR}')
print(f'Plot output directory: {PLOT_DIR}')
for path in sorted(OUTPUT_DIR.glob('*')):
    print(path)
for path in sorted(PLOT_DIR.glob('*.html')):
    print(path)

CSV output directory: tuning_analysis_outputs/glue_high_round_monitoring
Plot output directory: plots_epoch_round_tuning/glue_high_round_monitoring
tuning_analysis_outputs/glue_high_round_monitoring/qnli_latest_rounds.csv
tuning_analysis_outputs/glue_high_round_monitoring/qnli_round_metrics_raw.csv
tuning_analysis_outputs/glue_high_round_monitoring/qnli_round_metrics_summary.csv
